# 04 - Customer 360 Feature Engineering

**Goal:** create one analysis-ready row per canonical customer. Features combine transaction history with clearly labeled behavioral and synthetic marketing signals.

Monetary values use Brazilian reais (BRL), matching the Olist source.

In [1]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 120)
sns.set_theme(style="whitegrid", palette="Set2")


def find_project_root() -> Path:
    """Find the project root whether Jupyter starts here or in notebooks/."""
    for candidate in (Path.cwd(), Path.cwd().parent):
        if (candidate / "requirements.txt").exists() and (candidate / "notebooks").is_dir():
            return candidate.resolve()
    raise FileNotFoundError("Start Jupyter from the project root or notebooks directory.")


ROOT = find_project_root()
PROCESSED = ROOT / "data" / "processed"
MODELS = ROOT / "models"
PROCESSED.mkdir(parents=True, exist_ok=True)
MODELS.mkdir(parents=True, exist_ok=True)

print("Project root located.")

Project root located.


## 1. Load warehouse extracts and define the analysis snapshot

In [3]:
orders = pd.read_csv(PROCESSED / "fact_orders.csv", parse_dates=["purchase_date"])
customers = pd.read_csv(PROCESSED / "dim_customer.csv")
products = pd.read_csv(PROCESSED / "dim_product.csv")
web = pd.read_csv(PROCESSED / "fact_web_activity.csv", parse_dates=["event_time"])
customer_reviews = pd.read_csv(PROCESSED / "fact_customer_reviews.csv")
campaigns = pd.read_csv(PROCESSED / "fact_campaign.csv")

valid_orders = orders.loc[~orders["order_status"].isin(["canceled", "unavailable"])].copy()
snapshot_date = valid_orders["purchase_date"].max().normalize() + pd.Timedelta(days=1)
print(f"Feature snapshot: {snapshot_date.date()}")

Feature snapshot: 2018-09-04


## 2. Transaction and RFM features

Average order value is calculated from customer revenue divided by distinct orders. Taking the mean of item prices would incorrectly describe an item average rather than a basket average.

In [4]:
order_product = valid_orders.merge(
    products[["product_id", "category_name_english"]],
    on="product_id",
    how="left",
    validate="many_to_one",
)
transaction_features = (
    order_product.groupby("customer_id", as_index=False)
    .agg(
        total_orders=("order_id", "nunique"),
        total_spend=("revenue", "sum"),
        total_freight=("freight_value", "sum"),
        first_purchase_date=("purchase_date", "min"),
        last_purchase_date=("purchase_date", "max"),
        number_of_products=("product_id", "nunique"),
        category_diversity=("category_name_english", "nunique"),
    )
)
transaction_features["avg_order_value"] = transaction_features["total_spend"] / transaction_features["total_orders"]
transaction_features["recency_days"] = (snapshot_date - transaction_features["last_purchase_date"]).dt.days
transaction_features["frequency"] = transaction_features["total_orders"]
transaction_features["monetary"] = transaction_features["total_spend"]
transaction_features["customer_age_days"] = (
    transaction_features["last_purchase_date"] - transaction_features["first_purchase_date"]
).dt.days.clip(lower=1)

category_value = (
    order_product.groupby(["customer_id", "category_name_english"], as_index=False)["revenue"].sum()
    .sort_values(["customer_id", "revenue", "category_name_english"], ascending=[True, False, True])
    .drop_duplicates("customer_id")
    .rename(columns={"category_name_english": "favorite_category"})
    [["customer_id", "favorite_category"]]
)
transaction_features = (
    transaction_features.merge(category_value, on="customer_id", how="left", validate="one_to_one")
    .merge(customers[["customer_id", "city", "state"]], on="customer_id", how="left", validate="one_to_one")
)

## 3. Behavioral engagement features

In [5]:
web_features = (
    web.groupby("customer_id", as_index=False)
    .agg(
        sessions=("user_session", "nunique"),
        views=("event_type", lambda values: values.eq("view").sum()),
        cart_additions=("event_type", lambda values: values.eq("cart").sum()),
        web_purchases=("event_type", lambda values: values.eq("purchase").sum()),
    )
)
web_features["raw_engagement"] = (
    web_features["views"] + 3 * web_features["cart_additions"]
    + 5 * web_features["web_purchases"] + 2 * web_features["sessions"]
)
max_engagement = max(float(web_features["raw_engagement"].max()), 1.0)
web_features["web_engagement_score"] = (
    100 * np.log1p(web_features["raw_engagement"]) / np.log1p(max_engagement)
).round(2)
web_features["web_conversion_rate"] = (
    web_features["web_purchases"] / web_features["sessions"].replace(0, np.nan)
).fillna(0).clip(0, 1)
web_features = web_features.drop(columns="raw_engagement")

## 4. Customer feedback and synthetic campaign features

In [6]:
review_features = (
    customer_reviews.groupby("customer_id", as_index=False)
    .agg(
        avg_review_score=("rating", "mean"),
        reviews_given=("review_id", "count"),
        low_rating_count=("rating", lambda values: values.le(2).sum()),
    )
)

campaign_features = (
    campaigns.groupby("customer_id", as_index=False)
    .agg(
        campaigns_received=("campaign_id", "count"),
        campaign_opens=("opened", "sum"),
        campaign_clicks=("clicked", "sum"),
        campaign_conversions=("converted", "sum"),
        campaign_revenue=("revenue_generated", "sum"),
    )
)
campaign_features["campaign_open_rate"] = (
    campaign_features["campaign_opens"] / campaign_features["campaigns_received"].replace(0, np.nan)
)
campaign_features["campaign_conversion_rate"] = (
    campaign_features["campaign_conversions"] / campaign_features["campaigns_received"].replace(0, np.nan)
)

## 5. Assemble and validate the feature store

In [7]:
features = (
    transaction_features
    .merge(web_features, on="customer_id", how="left", validate="one_to_one")
    .merge(review_features, on="customer_id", how="left", validate="one_to_one")
    .merge(campaign_features, on="customer_id", how="left", validate="one_to_one")
)

zero_fill_columns = [
    "sessions", "views", "cart_additions", "web_purchases", "web_engagement_score",
    "web_conversion_rate", "reviews_given", "low_rating_count", "campaigns_received",
    "campaign_opens", "campaign_clicks", "campaign_conversions", "campaign_revenue",
    "campaign_open_rate", "campaign_conversion_rate",
]
features[zero_fill_columns] = features[zero_fill_columns].fillna(0)
features["avg_review_score"] = features["avg_review_score"].fillna(features["avg_review_score"].median())

assert features["customer_id"].is_unique, "Feature store must have one row per canonical customer."
assert features["total_orders"].ge(1).all()
assert features["total_spend"].ge(0).all()
assert features["recency_days"].ge(0).all()
assert np.allclose(features["avg_order_value"], features["total_spend"] / features["total_orders"])

features.to_csv(PROCESSED / "customer_feature_store.csv", index=False)
print(f"Saved {len(features):,} canonical customer profiles with {features.shape[1]} features.")
display(features.head())

Saved 94,983 canonical customer profiles with 32 features.


,customer_id,total_orders,total_spend,total_freight,first_purchase_date,last_purchase_date,number_of_products,category_diversity,avg_order_value,recency_days,frequency,monetary,customer_age_days,favorite_category,city,state,sessions,views,cart_additions,web_purchases,web_engagement_score,web_conversion_rate,avg_review_score,reviews_given,low_rating_count,campaigns_received,campaign_opens,campaign_clicks,campaign_conversions,campaign_revenue,campaign_open_rate,campaign_conversion_rate
0,0000366f3b9a7992bf8c76cfdf3221e2,1,129.90,12.00,2018-05-10 10:56:27,2018-05-10 10:56:27,1,1,129.90,116,1,129.90,1,bed_bath_table,cajamar,SP,1.0,1.0,0.0,0.0,16.12,0.0,5.0,1.0,0.0,1.0,1.0,1.0,1.0,158.46,1.0,1.0
1,0000b849f77a49e4a4ce2b2a4ca5be3f,1,18.90,8.29,2018-05-07 11:11:27,2018-05-07 11:11:27,1,1,18.90,119,1,18.90,1,health_beauty,osasco,SP,1.0,1.0,0.0,0.0,16.12,0.0,4.0,1.0,0.0,0.0,0.0,0.0,0.0,0.00,0.0,0.0
2,0000f46a3911fa3c0805444483337064,1,69.00,17.22,2017-03-10 21:05:03,2017-03-10 21:05:03,1,1,69.00,542,1,69.00,1,stationery,sao jose,SC,5.0,5.0,0.0,0.0,32.23,0.0,3.0,1.0,0.0,1.0,0.0,0.0,0.0,0.00,0.0,0.0
3,0000f6ccb0745a6a4b88665a16c9f078,1,25.99,17.63,2017-10-12 20:29:41,2017-10-12 20:29:41,1,1,25.99,326,1,25.99,1,telephony,belem,PA,3.0,7.0,0.0,0.0,30.68,0.0,4.0,1.0,0.0,0.0,0.0,0.0,0.0,0.00,0.0,0.0
4,0004aac84e0df4da2b147fca70cf8255,1,180.00,16.89,2017-11-14 19:45:42,2017-11-14 19:45:42,1,1,180.00,293,1,180.00,1,telephony,sorocaba,SP,0.0,0.0,0.0,0.0,0.00,0.0,5.0,1.0,0.0,1.0,1.0,0.0,0.0,0.00,1.0,0.0


## Feature limitations

Clickstream dates occur after the Olist transaction period and identity links are simulated. Therefore, web and campaign fields enrich dashboard profiles but are deliberately excluded from the time-based churn and CLV training features in Notebook 05. This prevents temporal leakage and overclaiming.